MODEL (baseline xgboost on all features)

In [ ]:
#pip install xgboost scikit-learn  (for macbook also: brew install libomp)
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from cohort import getCohort
from features import getFeatures, getGroupColumns

#5 folds so test and validation are each 1/5 = 20%
folds = 5
randomState = 42
#above this probability we predict failure
threshold = 0.5

FEATURES

In [ ]:
features = getFeatures(getCohort())
groupColumns = getGroupColumns(features)
for group, columns in groupColumns.items():
    print(group, len(columns))
print(f"extubations: {len(features)}, failure rate: {features['failure'].mean():.3f}")

In [ ]:
# double double check as its a bit much
print(features.isna().mean().sort_values(ascending=False).head(60).to_string())

SPLIT on person_id 

In [ ]:
#same failure rate but still split on patient
splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=randomState)
splits = list(splitter.split(X=features, y=features["failure"], groups=features["person_id"]))
trainIndex = splits[0][0]
testIndex = splits[0][1]
train = features.iloc[trainIndex]
test = features.iloc[testIndex]

trainSplits = list(splitter.split(X=train, y=train["failure"], groups=train["person_id"]))
fitIndex = trainSplits[0][0]
validationIndex = trainSplits[0][1]
trainFit = train.iloc[fitIndex]
validation = train.iloc[validationIndex]
print(f"train: {len(trainFit)} failure rate {trainFit['failure'].mean():.3f}, validation: {len(validation)} failure rate {validation['failure'].mean():.3f}, test: {len(test)} failure rate {test['failure'].mean():.3f}")

MODEL. TUNE LATER!!

In [ ]:
def getModel(scalePosWeight):
    return XGBClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=15,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_lambda=5,
        scale_pos_weight=scalePosWeight,
        early_stopping_rounds=50,
        enable_categorical=True,
        tree_method="hist",
        eval_metric="logloss",
        random_state=randomState,
    )

BASELINE all features together

In [ ]:
columns = groupColumns["all"]
scalePosWeight = (1 - trainFit["failure"].mean()) / trainFit["failure"].mean()
print(f"scale_pos_weight: {scalePosWeight:.1f}")
model = getModel(scalePosWeight)
model.fit(trainFit[columns], trainFit["failure"].astype(int), eval_set=[(validation[columns], validation["failure"].astype(int))], verbose=False)
print(f"trees used: {model.best_iteration + 1}")

for setName, data in [("train", trainFit), ("validation", validation)]:
    probability = model.predict_proba(data[columns])[:, 1]
    prediction = (probability >= threshold).astype(int)
    print(f"{setName}:")
    print(f"auroc: {roc_auc_score(data['failure'], probability):.3f}")
    print(f"auprc: {average_precision_score(data['failure'], probability):.3f} (random {data['failure'].mean():.3f})")
    print(f"accuracy: {accuracy_score(data['failure'], prediction):.3f} (always success {1 - data['failure'].mean():.3f})")
    print(f"precision: {precision_score(data['failure'], prediction, zero_division=0):.3f}")
    print(f"recall: {recall_score(data['failure'], prediction):.3f}")
    print(f"f1: {f1_score(data['failure'], prediction):.3f}")
    trueNegative, falsePositive, falseNegative, truePositive = confusion_matrix(data['failure'], prediction).ravel()
    print(f"specificity: {trueNegative / (trueNegative + falsePositive):.3f}")
    print(f"confusion matrix: tn {trueNegative}, fp {falsePositive}, fn {falseNegative}, tp {truePositive}")
    print()

In [ ]:
#most important features
importance = pd.Series(model.feature_importances_, index=columns).sort_values(ascending=False)
print(importance.head(20).to_string())